<a href="https://colab.research.google.com/github/laoneya31/IAT360-Group-18/blob/main/notebooks/Daksh_Setup_Baseline_BatchSize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><h1><b>Fruit Freshness Detection with YOLO</b></h1></center>
<center>IAT 360 Group 18: training environment, baseline and batch size (Daksh)</center>

This notebook covers my part of the project:

| Task | Section |
|---|---|
| Training environment: YOLO repo cloned, Colab GPU, shared training script | 1 |
| Quick test training on a small subset, to catch format errors early | 2 to 4 |
| Train and save the baseline, and share its results | 5 |
| Tune batch size at 3 values and compare with the baseline | 6 |
| Export YOLO plots and confusion matrices into one folder | 7 |
| Shortcomings and challenges notes for the report | 8 |

**How it switches between real and practice data.** Sections 5 to 7 use the group dataset as soon as Philip's
`fruit_freshness.zip` is on Drive. Until then they run a **rehearsal** on the small test set from Section 2, with tiny
settings, only to prove the code works. Rehearsal numbers are not results and must not go in the report.

## 1. Environment

Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**. The shared Drive folder must be in your My Drive: right-click `IAT360-Group 18` in *Shared with me* and choose **Organize > Add shortcut**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Clone our repo and the official YOLO repo. YOLO is pinned to one release so all four members train with identical code.

In [ ]:
import os, sys
REPO = "/content/IAT360-Group-18"
YOLO_TAG = "v8.4.204"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/laoneya31/IAT360-Group-18 {REPO}
# The folder must not be called "ultralytics": /content is on sys.path, so it would hide the real package.
if not os.path.exists("/content/yolo_repo"):
    !git -c advice.detachedHead=false clone -q --depth 1 --branch {YOLO_TAG} https://github.com/ultralytics/ultralytics /content/yolo_repo
%pip install -q /content/yolo_repo kagglehub
%cd {REPO}

In [ ]:
import ultralytics
ultralytics.checks()
from ultralytics import YOLO

sys.path.insert(0, f"{REPO}/src")
import fruit_yolo as fy
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display

fy.check_config()
print("Team Drive folder:", fy.find_team_drive())
print("Baseline settings:", fy.BASELINE)

## 2. Dataset for the quick test

The group dataset has six classes: `fresh_apple`, `fresh_banana`, `fresh_orange`, `rotten_apple`, `rotten_banana`,
`rotten_orange`. To test the pipeline before it is ready, I build a small multi-fruit set in the same format from the
Kaggle [Fruits fresh and rotten](https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification)
images. Each single-fruit photo has a plain background, so the fruit can be cut out and pasted on a new background, which
also gives an exact bounding box. This set is regenerated each run and is never committed to GitHub.

In [ ]:
import glob, hashlib, kagglehub, random

kaggle_root = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
source_root = f"{kaggle_root}/dataset/train"
folders = ["freshapples", "freshbanana", "freshoranges", "rottenapples", "rottenbanana", "rottenoranges"]
sources = {i: sorted(glob.glob(f"{source_root}/{name}/*")) for i, name in enumerate(folders)}
for i, name in enumerate(folders):
    print(f"{fy.CLASS_NAMES[i]:<15} <- {name:<14} {len(sources[i])} photos")

This download holds a second copy of everything in a nested `dataset/dataset/` folder. Checking whether it is a true duplicate, because a recursive search would pull the same photos into both train and test.

In [ ]:
def md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

sample = random.Random(0).sample(sources[0], 200)
nested = [p.replace("/dataset/train/", "/dataset/dataset/train/") for p in sample]
same = sum(os.path.exists(n) and md5(n) == md5(p) for p, n in zip(sample, nested))
print(f"{same} of {len(sample)} sampled files are byte-identical in the nested copy")

Build the test set. Source photos are split between train, val and test first, so the same photo cannot appear in two splits.

In [ ]:
SMOKE = "/content/datasets/smoke"
SMOKE_CONFIG = "/content/smoke.yaml"
rng = random.Random(0)
subset = {cid: rng.sample(files, 60) for cid, files in sources.items()}

if not os.path.exists(SMOKE):
    print("boxes written:", fy.build_composites(subset, SMOKE, {"train": 120, "val": 30, "test": 30}, seed=0))
fy.write_config(SMOKE, SMOKE_CONFIG)
print(open(SMOKE_CONFIG).read())

Visualising some scenes with their labels, drawn back from the YOLO `.txt` files. This confirms the labels survive the round trip to normalised coordinates.

In [ ]:
import cv2

fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
for ax, path in zip(axes, sorted(glob.glob(f"{SMOKE}/images/train/*.jpg"))[:4]):
    image = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    label = path.replace("/images/", "/labels/").replace(".jpg", ".txt")
    for c, x1, y1, x2, y2 in fy._yolo_boxes(label, width, height):
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(image, fy.CLASS_NAMES[c], (int(x1), max(14, int(y1) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    ax.imshow(image); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Format check

The tutorial's missing-file check, extended. `validate_dataset` also catches the errors that do not crash YOLO but
silently damage training: class ids out of range, pixel instead of normalised coordinates, polygon exports, and the same
image in two splits. Every member can run it on any dataset folder before training.

In [ ]:
report = fy.validate_dataset(SMOKE)
fy.assert_valid(report)

A check is only useful if it fails when it should. Here I break a copy of the test set on purpose, once for each kind of mistake we could plausibly make.

In [ ]:
import shutil

BROKEN = "/content/datasets/broken"
shutil.rmtree(BROKEN, ignore_errors=True)
shutil.copytree(SMOKE, BROKEN)
labels = sorted(glob.glob(f"{BROKEN}/labels/train/*.txt"))
images = sorted(glob.glob(f"{BROKEN}/images/train/*.jpg"))

open(labels[0], "w").write("6 0.5 0.5 0.2 0.2\n")               # class id that does not exist
open(labels[1], "w").write("0 320 240 50 50\n")                 # pixels, not normalised
open(labels[2], "w").write("0 0.1 0.1 0.2 0.2 0.3 0.3\n")       # polygon export
os.remove(labels[3])                                             # label lost
os.remove(images[4])                                             # image lost
shutil.copy(images[5], f"{BROKEN}/images/val/leaked.jpg")        # train image copied into val
shutil.copy(labels[5], f"{BROKEN}/labels/val/leaked.txt")

broken = fy.validate_dataset(BROKEN, near_duplicates=False)
assert len(broken["errors"]) == 6, broken["errors"]
print("\nAll 6 planted mistakes were caught.")

## 4. Quick test training

Two epochs' worth of confidence before anyone spends GPU hours: if this runs, the folder layout, labels, config and
training code all work together.

We use the same pretrained YOLOv8 approach as the course tutorial. The smaller `yolov8n` and a low image size are used
here only to keep this test fast.

In [ ]:
smoke_weights = fy.train_run("smoke_test", "Daksh", data=SMOKE_CONFIG, rehearsal=True)
smoke = fy.evaluate("smoke_test", smoke_weights, data=SMOKE_CONFIG, rehearsal=True)
print({k: round(v, 3) for k, v in smoke.items() if isinstance(v, float)})

## 5. Baseline model

**Why the baseline writes out its optimizer.** With YOLO's default `optimizer=auto`, the training log says it is
*"ignoring lr0"*, so Philip's learning-rate runs would all train the same model. For a dataset our size, `auto` picks
AdamW with `lr0=0.001`, `momentum=0.9` and `warmup_bias_lr=0`, so `BASELINE` sets those explicitly. Behaviour stays the same and `lr0` starts working.

**Other settings** are YOLO defaults (`imgsz=640`, `batch=16`) with `yolov8s` from the course tutorial. Epochs are 50
rather than 100 with `patience=20`, to fit 13+ runs on free Colab GPUs before the deadline.

In [ ]:
REAL = fy.fetch_dataset()
if REAL:
    fy.require_gpu()
    fy.assert_valid(fy.validate_dataset())
    DATA, ROOT, REHEARSAL = fy.CONFIG, fy.DATASET, False
else:
    DATA, ROOT, REHEARSAL = SMOKE_CONFIG, SMOKE, True
    print("\n*** REHEARSAL: group dataset not ready. Numbers below only prove the code runs. Do not report them. ***")
SIZE = fy.REHEARSAL["imgsz"] if REHEARSAL else fy.BASELINE["imgsz"]
print("Results go to:", fy.output_root(REHEARSAL))

In [ ]:
baseline_weights = fy.train_run("baseline", "Daksh", data=DATA, rehearsal=REHEARSAL)

### Evaluation metrics

Following the tutorial's advice on choosing metrics: our use case is finding rotten fruit before it spreads, so a missed
rotten fruit costs more than a false alarm. **Recall on the rotten classes** matters most, alongside mAP for overall quality.

- `accuracy`: correct boxes / (correct + wrong class + missed + false alarms), from the confusion matrix at confidence 0.25, the threshold a deployed system would use. Detection has no true negatives, so this is the closest equivalent. mAP is measured separately at YOLO's default low confidence, because it sweeps every threshold.
- `rotten_recall`: average recall of the three rotten classes.
- `rotten_flag_recall`: share of rotten fruit flagged as rotten of *any* type. Calling a rotten apple a rotten orange still gets it removed from the bin.

In [ ]:
baseline = fy.evaluate("baseline", baseline_weights, data=DATA, rehearsal=REHEARSAL)
display(pd.DataFrame([{k: baseline[k] for k in fy.METRIC_COLUMNS}], index=["baseline"]).round(3))
display(pd.DataFrame(baseline["per_class"]).T.round(3))
print({k: baseline[k] for k in ("rotten_called_fresh", "rotten_missed", "false_alarms")})

Showing the confusion matrix and training curves that YOLO saved.

In [ ]:
figures = fy.output_root(REHEARSAL) / "figures"
display(Image.open(figures / "baseline_val" / "confusion_matrix_normalized.png").resize((720, 540)))
display(Image.open(figures / "baseline" / "results.png").resize((1000, 500)))

Is the model underfitting or overfitting? The train and validation losses give the evidence.

In [ ]:
fy.fit_report("baseline", rehearsal=REHEARSAL)

### Correct and incorrect examples

Green boxes are the true labels, orange boxes are predictions.

In [ ]:
scored = fy.score_images(baseline_weights, root=ROOT, split="val", imgsz=SIZE)
wrong = sum(bool(s["problems"]) for s in scored)
print(f"{len(scored) - wrong} images fully correct, {wrong} with at least one mistake")
fy.show_examples(scored, correct=True, n=3, save_to=figures / "baseline" / "examples_correct.png"); plt.show()
fy.show_examples(scored, correct=False, n=3, save_to=figures / "baseline" / "examples_incorrect.png"); plt.show()

## Prediction

Running the saved model on a real photo that it never trained on, as the tutorial does.

In [ ]:
photos = kagglehub.dataset_download("mbkinaci/fruit-images-for-object-detection")
photo = sorted(glob.glob(f"{photos}/test_zip/test/mixed_*.jpg"))[0]
result = YOLO(str(baseline_weights)).predict(photo, imgsz=SIZE, conf=0.25, verbose=False)[0]
print(result.verbose() or "no fruit detected")
display(Image.fromarray(result.plot()[..., ::-1]))

## 6. Batch size tuning

Baseline `batch=16`, tuned at **8, 32 and 64**.

**What to expect.** YOLO accumulates gradients up to a nominal batch of 64 (`nbs=64`), so the weights update about every
64 images whichever batch we choose: 8 x 8, 16 x 4, 32 x 2, 64 x 1. Batch size therefore mostly changes BatchNorm
statistics and speed, not the size of each update, so the differences may be small. The table records the effective batch
for each run so this can be checked rather than assumed.

In [ ]:
OWNER = "Daksh"
PARAM, VALUES = fy.TUNING_PLAN[OWNER]
RUNS = [f"{PARAM}_{v}" for v in VALUES]

for name, value in zip(RUNS, VALUES):
    weights = fy.train_run(name, OWNER, overrides={PARAM: value}, data=DATA, rehearsal=REHEARSAL)
    fy.evaluate(name, weights, data=DATA, rehearsal=REHEARSAL)

In [ ]:
table = fy.compare_runs(names=["baseline"] + RUNS, rehearsal=REHEARSAL)
display(table)
title = "Batch size: mAP@0.5:0.95 on validation" + (" (REHEARSAL, not results)" if REHEARSAL else "")
fy.plot_comparison(table, title, save_to=figures / "tuning_batch.png")
plt.show()
for name in RUNS:
    print(name, fy.fit_report(name, rehearsal=REHEARSAL))

### My results

*TODO after the real run: which batch size won, by how much, and whether the gap is larger than run-to-run noise.*

## 7. Figures for the slides

Every run's plots and confusion matrices are in one folder per run, so all slides use the same figures. On real runs this is the shared Drive folder `IAT360-Group 18/results/figures`.

In [ ]:
for folder in sorted(p for p in figures.iterdir() if p.is_dir()):
    print(f"{folder.name:<20}", len(list(folder.glob("*.png"))), "plots")

## 8. Notes for my report section

See `report/Daksh_Shortcomings_and_Challenges.md`. Update it with real numbers after Sections 5 and 6 have run on the group dataset.